In [28]:
import os
import csv
from Bio import SeqIO

First we will parse genbank file, filter sequences with ambiguous characters

In [20]:
recs = list(SeqIO.parse("raw/Bastrovirus_22102026.gb", "genbank"))


#list with ambiguous nucleotides
ambig_nt = ['n',  # A, T, G, C 
            'r', # Purine (A or G)
            'y', # Pyrimidine (T or C)
            'k', # Keto (G or T)
            'm', # Amino (A or C)
            's', # Strong interaction (3 H bonds) (G or C)
            'w', # Weak interaction (2 H bonds) (A or T)
            'b', # Not A (C or G or T)
            'd', # Not C (A or G or T)
            'h', # Not G (A or C or T)
            'v' # Not T or U (A or C or G)
            ]

recs_noamb = []
total_amb = 0
for rec in recs:
    for nt in ambig_nt:
        total_amb += rec.seq.lower().count(nt)
    if total_amb/len(rec.seq)< 0.08:
        recs_noamb.append(rec)
    else:
        print(f"{rec.name} has {total_amb} ({round(100*total_amb/len(rec.seq),2)}%) ambiguous nucleotides")
print(f"There are {len(recs_noamb)} sequences with <1% ambiguous nucleotides")

SeqIO.write(recs_noamb, "raw/Bastrovirus_22102026_lessamb.gb", "genbank")
SeqIO.write(recs_noamb, "raw/Bastrovirus_22102026_lessamb.fna", "fasta")

OQ802730 has 547 (9.1%) ambiguous nucleotides
OQ802729 has 547 (9.31%) ambiguous nucleotides
OQ802713 has 547 (9.9%) ambiguous nucleotides
OR439362 has 547 (9.79%) ambiguous nucleotides
OR552431 has 566 (9.98%) ambiguous nucleotides
OR552430 has 570 (9.77%) ambiguous nucleotides
OQ599876 has 570 (8.15%) ambiguous nucleotides
ON304005 has 570 (9.68%) ambiguous nucleotides
OM953852 has 743 (12.94%) ambiguous nucleotides
MT568535 has 743 (14.17%) ambiguous nucleotides
MT766313 has 743 (14.11%) ambiguous nucleotides
LC577872 has 743 (12.44%) ambiguous nucleotides
LC577871 has 743 (13.42%) ambiguous nucleotides
LC577870 has 743 (14.26%) ambiguous nucleotides
MG693175 has 743 (13.08%) ambiguous nucleotides
NC_035758 has 743 (11.72%) ambiguous nucleotides
KU318321 has 743 (11.72%) ambiguous nucleotides
KU318320 has 744 (11.81%) ambiguous nucleotides
KU318319 has 746 (12.35%) ambiguous nucleotides
KU318318 has 746 (12.26%) ambiguous nucleotides
KU318317 has 746 (11.77%) ambiguous nucleotides
K

28

Get coding sequences for future hmmsearch

In [24]:
#======= Analyses CDS feature which annotation does not correspond to target CDS =======
def analyze_cds(record, feature, transl_table=1):
    """
    Extracts translation and coordinates from a CDS feature.
    Returns a dict with details, including internal stop count.
    """
    
    codon_start = int(feature.qualifiers.get("codon_start", [1])[0])
    offset = codon_start - 1


    # Get parts – if not a compound location, treat as single part
    if len(feature.location.parts) > 1:
        parts = feature.location.parts
    else:
        parts = [feature.location]

    
    candidates = []
    for idx, part in enumerate(parts):
        raw_seq = part.extract(record.seq)
        # codon_start offset only applies to the first part
        if idx == 0:
            cds_seq = raw_seq[offset:]
        else:
            cds_seq = raw_seq

        # Translate, include stop codon
        translation = cds_seq.translate(table=transl_table, to_stop=False)
        annotated_translation = feature.qualifiers.get("translation", [""])[0]
        
        # Count internal stops (* not at last position)
        stops = [i for i, aa in enumerate(translation) if aa == "*" and i != len(translation)-1]
        internal_stop_count = len(stops)
        
        if translation and translation[-1] == "*":
            translation = translation[:-1]

        if len(parts) == 1 and translation != annotated_translation:
            print(f"For {record.id} the annotated translation does not correspond to actual translation of coding sequence")
            print("Actual translation: " + translation)
            print("Annotated translation: " + annotated_translation)


        start = int(part.start)
        end = int(part.end)
        
        cand = {
            "record_name": record.name,
            "accession": record.id,
            "location": str(part),                # individual part location
            "codon_start": str(codon_start if idx == 0 else 1),
            "start": start,
            "end": end,
            "strand": part.strand,
            "gene": feature.qualifiers.get("gene", ["unknown"])[0],
            "product": feature.qualifiers.get("product", ["unknown"])[0],
            "transl_table": transl_table,
            "raw_nucleotide": str(raw_seq),
            "cds_sequence": str(cds_seq),
            "translation": str(translation),
            "translation_annot": str(annotated_translation),
            "internal_stop_count": internal_stop_count,
            "is_clean": internal_stop_count == 0,
            "part_index": idx,   # optional, for debugging
        }
        candidates.append(cand)
    return candidates

#======= Validation and correction of frame in CDS feature which annotation does not correspond to target CDS =======
def correct_cds_frame(cds_info, max_stops=2):
    """
    Attempt to correct the translation frame of a CDS.
    Tries all 3 reading frames on the raw nucleotide sequence.
    Returns the best translation and the offset to apply.

    Parameters:
    cds_info - dictionary with information of CDS feature, output of analyze_cds funciton
    max_stops - maximal number of stop codons
    """
    raw = cds_info["raw_nucleotide"]  # use the untrimmed raw sequence
    transl_table = cds_info["transl_table"]
    
    best = None
    best_offset = None
    
    for frame in range(3):  # 0, 1, 2
        seq = Seq(raw[frame:])
        
        trans = seq.translate(table=transl_table, to_stop=False)
        # count internal stops
        internal_stops = [i for i, aa in enumerate(trans) if aa == "*" and i != len(trans)-1]
        stop_count = len(internal_stops)
        
        # We prefer fewer stops, and if equal, longer length
        if stop_count <= max_stops:
            if best is None or (stop_count < best["stop_count"]) or \
               (stop_count == best["stop_count"] and len(trans) > len(best["translation"])):
                best = {
                    "translation": str(trans),
                    "stop_count": stop_count,
                    "frame_offset": frame,  # number of bases to skip from the raw start
                    "nucleotide_used": str(seq),
                }
                best_offset = frame
    return best, best_offset
# ======= write CDS with no target or conflicting annotation
def write_fasta_and_mapping(candidates, fasta_file, mapping_file):
    """
    Writes two files:
      - fasta_file: amino acid sequences with header containing accession and some metadata.
      - mapping_file: TSV (or CSV) with detailed coordinates/strand per candidate.
    """
    with open(fasta_file, "w") as f_fasta, open(mapping_file, "w", newline="") as f_map:
        writer = csv.writer(f_map, delimiter="\t")
        writer.writerow(["candidate_id",
            "record_name", "accession", "start", "end", "codon_start","strand",
            "gene", "product", "is_corrected", "correction_offset", "original_translation"
        ])
        for idx, c in enumerate(candidates, 1):
            # FASTA header: include ID and some quick info
            cid = f"cand_{idx:04d}"
            header = f">{cid}|{c['accession']}|{c['gene']}|start={c['start']}|end={c['end']}|codon_start={c['codon_start']}|strand={c['strand']}"
            f_fasta.write(header + "\n")
            trans = c['translation']
            for i in range(0, len(trans), 60):
                f_fasta.write(trans[i:i+60] + "\n")
            
            # Write mapping line
            writer.writerow([
                cid,
                c['record_name'],
                c['accession'],
                c['start'],
                c['end'],
                c['codon_start'],
                c['strand'],
                c['gene'],
                c['product'],
                c.get('is_corrected', False),
                c.get('correction_frame_offset', ''),
                c.get('original_translation', '')
            ])

In [29]:
cds_candidates = []
cds_counts = []
for rec in recs_noamb:
    cds_count = 0
    
    rec_name = rec.id.split(".")[0]

    for feature in rec.features:
        if feature.type == 'CDS':
            cds_count +=1
            #print(feature)
            cds_infos = analyze_cds(rec, feature)
            for cds_info in cds_infos:
                if not cds_info["is_clean"]:
                    corrected, offset = correct_cds_frame(cds_info, max_stops=1)
                    if corrected is not None:
                        cds_info["is_corrected"] = True
                        cds_info["corrected_translation"] = corrected["translation"]
                        cds_info["original_translation"] = cds_info["translation"]
                        cds_info["translation"] = corrected["translation"]
                        cds_info["correction_frame_offset"] = offset
                        cds_info["codon_start"] = int(cds_info["codon_start"]) + offset
                    else:
                        cds_info["is_corrected"] = False
                        cds_info["correction_failed"] = True
                cds_candidates.append(cds_info)
    print(f"{rec.name} has {cds_count} annotated CDS")
    cds_counts.append(cds_count)
    print("_______")

out_folder = "cds"
out_file_name_temp = "bastro"
#os.mkdir(out_folder)
# Write all CDS

fasta_cand = os.path.join(out_folder, out_file_name_temp + '_cds.faa')
mapping_cand = os.path.join(out_folder, out_file_name_temp + '_cds.tsv')
write_fasta_and_mapping(cds_candidates, fasta_cand, mapping_cand)

For PV605168.1 the annotated translation does not correspond to actual translation of coding sequence
Actual translation: LDAVNGRGHTPKQLLWLQNPDVNHFQAVAVEGLLPLLPCDVIGSVDTNAFVGTLPAVLHAVNAVPGQVGALINHGPLKPQLIACVVHETAHVHCPPLDVAEVTTNVNRALTMPVSMVVSEGVQCAHHGMRVCDAILEHEVAPQNLEEVFLLLAVEHERNVQGIADSACHCTAEACAVLTFPVAALGVREGGGAGERSM
Annotated translation: MDAVNGRGHTPKQLLWLQNPDVNHFQAVAVEGLLPLLPCDVIGSVDTNAFVGTLPAVLHAVNAVPGQVGALINHGPLKPQLIACVVHETAHVHCPPLDVAEVTTNVNRALTMPVSMVVSEGVQCAHHGMRVCDAILEHEVAPQNLEEVFLLLAVEHERNVQGIADSACHCTAEACAVLTFPVAALGVREGGGAGERSM
For PV605168.1 the annotated translation does not correspond to actual translation of coding sequence
Actual translation: LHIPFVLNREEKEYLLKVLRCDLVFEDRVAHTHSMVRALHAFANHHAYRHCEGAIDIGGDFRNIKRRAVHVCGLVDNARDELRLQRAMVDQRPNLAGYCIHGVENCRERADKGICVHAAYDITWQQWQQAFDSHGLKVVDVWILEPEELFGRVTPPIYGIQMHRLANRWAMLLDDACAGYVHDDFEWRKYRETGGTIGPRFNIGCSPEGGWGPLRRYTLFRCVHPFRQITLRHPVDRNTVHVPTPAGHVISCPRSVWQNLLSWACSRADNKFDFSSLLAYARALRQRLTIGGQTVQEGWSQDPRQVHELCIACFVMAAAMRYTRTQNIAVAFAEL

Scan obtained CDS against PFAM